# PipeTwin · result.csv и профиль толщины
В режиме `demo_mode = True` ноутбук читает `tests/result.csv`, сохраняет `result_demo.csv` и строит графики из сохранённого файла.

Для реального результата выключите `demo_mode` и задайте путь к существующему входному CSV в `real_source`.

Для расчёта задайте `source` с исходной толщиной на физических сегментах, `use_precomputed_result = False` и горизонт. Скорость берётся из NORSOK M-506 в NeqSim или из явно заданного `manual_rate`. Для входного файла с несколькими участками выберите два через `submission_ids`. Порядковые записи сырого ВТД сначала требуют пространственной привязки к 10-метровым сегментам.

In [ ]:
from io import StringIO
from pathlib import Path

from IPython.display import SVG, display
import matplotlib.pyplot as plt

from backend.corrosion import co2_rate, remaining_thickness
from backend.submission import result_csv
from backend.web_api import ROOT, read_forecast

demo_mode = True
real_source = Path('data/result.csv')  # Укажите путь к своему готовому прогнозу.
source = ROOT / (Path('tests/result.csv') if demo_mode else real_source.expanduser())
output = ROOT / ('result_demo.csv' if demo_mode else 'result.csv')
use_precomputed_result = True
submission_ids = None  # Например, ('ID_первого', 'ID_второго')
passport = ROOT / ('tests/passport.csv' if demo_mode else 'data/passport.csv')
years = 5
manual_rate = None
conditions = dict(
    temperature_c=30, total_pressure_bar=20, co2_mol_percent=0.5,
    ph=5.0, flow_velocity_ms=1.0, diameter_mm=200,
)

if source.resolve() == output.resolve():
    raise ValueError('Укажите разные пути для входного и выходного CSV.')
if not source.is_file():
    raise FileNotFoundError(
        f'Входной CSV не найден: {source}. Укажите существующий файл в real_source '
        'или включите demo_mode для проверки на тестовых данных.'
    )
if passport and not passport.is_file():
    raise FileNotFoundError(f'Паспорт не найден: {passport}. Укажите существующий файл или passport = None.')
pipes = read_forecast(source.read_bytes())
if submission_ids is not None:
    by_id = {pipe['id']: pipe for pipe in pipes}
    missing = set(submission_ids) - by_id.keys()
    if missing:
        raise ValueError(f'Участки отсутствуют во входных данных: {sorted(missing)}')
    pipes = [by_id[pipe_id] for pipe_id in submission_ids]
if len(pipes) != 2:
    raise ValueError('Выберите два участка для сдачи через submission_ids.')
if any(pipe.get('coordinate_basis') == 'inspection_order' for pipe in pipes):
    raise ValueError('Сырые записи ВТД сначала нужно привязать к физическим сегментам.')
lengths = {pipe['id']: pipe['length_m'] for pipe in read_forecast(passport.read_bytes())} if passport else {}

In [ ]:
import math

rate = None
if not use_precomputed_result:
    if not 0 < years <= 100:
        raise ValueError('Горизонт должен быть больше 0 и не более 100 лет.')
    rate = co2_rate(**conditions) if manual_rate is None else manual_rate
    if not math.isfinite(rate) or rate < 0:
        raise ValueError('Скорость должна быть конечной и неотрицательной.')
    print(f'Скорость утонения: {rate:.4g} мм/год')
elif any(pipe.get('source_kind') == 'nominal' or any(row.get('replacement') for row in pipe['segments']) for pipe in pipes):
    raise ValueError('Паспорт или профиль с заменами требует расчёта. Укажите use_precomputed_result = False.')


def thickness_for_result(segment):
    if rate is None:
        return segment['thickness_pred']
    wall = segment.get('replacement', {}).get('wall_mm', segment['thickness_pred'])
    return remaining_thickness(wall, rate, years)


predictions = [
    {**pipe, 'length_m': lengths.get(pipe['id'], pipe.get('length_m')),
     'segments': [
         {'segment_id': row['segment_id'], 'thickness_pred': thickness_for_result(row)}
         for row in pipe['segments']
     ]}
    for pipe in pipes
]

In [ ]:
csv_text = result_csv(predictions)
output.write_bytes(csv_text.encode('utf-8'))
result = read_forecast(output.read_bytes())
if any(not pipe.get('length_m') for pipe in predictions):
    print('Сверьте количество сегментов с паспортом: для одного или обоих участков длина отсутствует.')
print(f'{output.name}: {len(result)} участка, {sum(len(pipe["segments"]) for pipe in result)} строк.')

In [ ]:
fig, axes = plt.subplots(len(result), 1, figsize=(10, 3.3 * len(result)), layout='constrained')
names = {pipe['id']: pipe.get('name') for pipe in pipes}
for axis, pipe in zip(axes, result):
    distance = [segment['segment_id'] * 10 for segment in pipe['segments']]
    thickness = [segment['thickness_pred'] for segment in pipe['segments']]
    axis.plot(distance, thickness, color='#2563a8', marker='o' if len(distance) < 100 else None)
    title = names[pipe['id']] or pipe['id']
    axis.set(title=f'{title} · result.csv', xlabel='Начало сегмента, м', ylabel='Толщина, мм')
    axis.grid(alpha=0.25)
with StringIO() as buffer:
    fig.savefig(buffer, format='svg')
    display(SVG(data=buffer.getvalue()))
plt.close(fig)